# MAS Conflict Resolution — PoC

All 4 strategies from `docs/solution-design.md` using `qwen3.5:2b-mlx`.

**Scenario:** ThroughputAgent wants ADVANCE_TO_APPROVAL on Loan Batch 123.
FairnessAgent wants HOLD_FOR_FAIRNESS_REVIEW on the same batch. Conflict.

**Pattern:** Single-string prompts (no message lists) — proven better for small models.
Each agent is a tiny LLM call returning JSON.

In [1]:
from langchain_ollama import ChatOllama
import json, re
from dataclasses import dataclass

llm = ChatOllama(model="qwen3.5:2b-mlx", reasoning=False, temperature=0)
print(f"LLM: {llm.model}")

LLM: qwen3.5:2b-mlx


In [2]:
def clean_json(text: str) -> dict:
    text = re.sub(r'^```(?:json)?\s*', '', text.strip())
    text = re.sub(r'\s*```$', '', text)
    match = re.search(r'\{.*\}', text, re.DOTALL)
    if match:
        return json.loads(match.group())
    raise ValueError(f"No JSON found in: {text[:200]}")

def ask(prompt: str) -> dict:
    return clean_json(llm.invoke(prompt).content)

# quick test
print(f"clean_json works: {ask('Output JSON: {\"ok\": true}')}")

clean_json works: {'ok': True}


In [3]:
@dataclass
class Plan:
    agent_id: str
    action: str
    target: str

p1 = Plan("ThroughputAgent", "ADVANCE_TO_APPROVAL", "Loan Batch 123")
p2 = Plan("FairnessAgent", "HOLD_FOR_FAIRNESS_REVIEW", "Loan Batch 123")

print(f"Conflict: {p1.target}")
print(f"  {p1.agent_id:<20s} -> {p1.action}")
print(f"  {p2.agent_id:<20s} -> {p2.action}")

Conflict: Loan Batch 123
  ThroughputAgent      -> ADVANCE_TO_APPROVAL
  FairnessAgent        -> HOLD_FOR_FAIRNESS_REVIEW


---
## Strategy 1: Policy-Based Resolution

In [4]:
print("=" * 55)
print("STRATEGY 1: POLICY-BASED")
print("=" * 55)

# Agent 1.1 - ConflictDetector
cd = ask(f'''Two actions on target "{p1.target}":
  {p1.agent_id}: {p1.action}
  {p2.agent_id}: {p2.action}
Same target + different action = conflict.
Output JSON: {{"conflict": true/false}}''')
print(f"  ConflictDetector: {cd}")

# deterministic policy logic (Python)
PRIORITY = {"HOLD_FOR_FAIRNESS_REVIEW": 10, "ADVANCE_TO_APPROVAL": 5}
POLICY_RULE = "FAIRNESS_CHECK_REQUIRED = True -> Fairness > Speed"

if PRIORITY.get(p2.action, 0) > PRIORITY.get(p1.action, 0):
    approved, denied = p2.agent_id, p1.agent_id
else:
    approved, denied = p1.agent_id, p2.agent_id

print(f"  Priority map: {PRIORITY}")
print(f"  Rule:         {POLICY_RULE}")
print(f"  >>> Approved: {approved}")
print(f"  >>> Denied:   {denied}")

# Agent 1.2 - AuditLogger
al = ask(f'''Resolution on {p1.target}:
  Approved: {approved}  Denied: {denied}
  Policy: {POLICY_RULE}  Priority: {PRIORITY}
Explain in 1-2 sentences why this is correct.
Output JSON: {{"explanation": "..."}}''')
print(f"  AuditLogger: {al.get('explanation', al)}")

STRATEGY 1: POLICY-BASED
  ConflictDetector: {'conflict': True}
  Priority map: {'HOLD_FOR_FAIRNESS_REVIEW': 10, 'ADVANCE_TO_APPROVAL': 5}
  Rule:         FAIRNESS_CHECK_REQUIRED = True -> Fairness > Speed
  >>> Approved: FairnessAgent
  >>> Denied:   ThroughputAgent
  AuditLogger: The decision is correct because the system prioritizes fairness over throughput when the fairness check flag is enabled, and the priority rules explicitly assign a higher value (10) to holding for fairness review than advancing to approval (5), ensuring that any potential unfairness in the loan batch takes precedence.


---
## Strategy 2: Hierarchical Resolution

In [5]:
print("=" * 55)
print("STRATEGY 2: HIERARCHICAL (SUPERVISOR OVERRIDE)")
print("=" * 55)

# Agent 2.1 - RiskAssessor for ThroughputAgent
rt = ask(f'''Agent: {p1.agent_id} wants {p1.action} on {p1.target}
Rate risk 1-10 (10 = highest risk). Consider system safety implications.
Output JSON: {{"risk_score": N, "description": "one sentence"}}''')
print(f"  {p1.agent_id} risk: {rt.get('risk_score', '?')} - {rt.get('description', '')}")

# Agent 2.2 - RiskAssessor for FairnessAgent
rf = ask(f'''Agent: {p2.agent_id} wants {p2.action} on {p2.target}
Rate risk 1-10 (10 = highest risk). Consider regulatory compliance implications.
Output JSON: {{"risk_score": N, "description": "one sentence"}}''')
print(f"  {p2.agent_id} risk: {rf.get('risk_score', '?')} - {rf.get('description', '')}")

# Python: lower risk -> approved; tie -> policy (Fairness > Speed)
r1 = rt.get("risk_score", 5)
r2 = rf.get("risk_score", 5)
if r2 < r1:
    approved, denied = p2.agent_id, p1.agent_id
    reason = f"lower risk ({r2} < {r1})"
elif r1 < r2:
    approved, denied = p1.agent_id, p2.agent_id
    reason = f"lower risk ({r1} < {r2})"
else:
    approved, denied = p2.agent_id, p1.agent_id
    reason = f"tie ({r1} == {r2}) - policy: Fairness > Speed"

# Agent 2.3 - Supervisor (binding decision with rationale)
sv = ask(f'''Supervisor decision on {p1.target}:
  Approved: {approved} (risk={r2 if approved==p2.agent_id else r1})
  Denied:   {denied}  (risk={r1 if denied==p1.agent_id else r2})
  Reason: {reason}
Write the official override order including rationale.
Output JSON: {{"order": "...", "rationale": "..."}}''')

print(f"\n  >>> Approved: {approved}")
print(f"  >>> Denied:   {denied}")
print(f"  Order: {sv.get('order', '?')}")
print(f"  Rationale: {sv.get('rationale', '?')}")

STRATEGY 2: HIERARCHICAL (SUPERVISOR OVERRIDE)
  ThroughputAgent risk: 8 - High rate risk score indicates significant potential for loan default or systemic instability if approval is granted.
  FairnessAgent risk: 8 - High regulatory risk due to potential non-compliance with fair lending practices if the review process is not explicitly documented and auditable.

  >>> Approved: FairnessAgent
  >>> Denied:   ThroughputAgent
  Order: Approved: FairnessAgent
  Rationale: The system tie-breaker policy explicitly prioritizes fairness over throughput when both agents have equal risk scores (8). Since the FairnessAgent was selected, its decision is upheld.


---
## Strategy 3: Negotiation (Bottom-Up)

In [6]:
print("=" * 55)
print("STRATEGY 3: NEGOTIATION")
print("=" * 55)

log = []
resolved = False
last = ""

for rnd in range(1, 3):
    print(f"\n--- Round {rnd} ---")

    prop = ask(f'''You={p1.agent_id}. Goal={p1.action} on {p1.target}.
    Other={p2.agent_id} wants {p2.action} on same target.
    {"Last round: " + log[-1] if log else "First round."}
    Propose a compromise.
    Output JSON: {{"proposal": "..."}}''')
    log.append(f"{p1.agent_id} proposes: {prop.get('proposal','?')}")
    print(f"  {p1.agent_id}: {prop.get('proposal','?')}")
    last = prop.get('proposal', '')

    resp = ask(f'''You={p2.agent_id}. Goal={p2.action} on {p1.target}.
    Other proposes: {last}
    Accept, reject, or counter?
    Output JSON: {{"decision": "accept/reject/counter", "message": "..."}}''')
    log.append(f"{p2.agent_id}: {resp.get('decision','?')} - {resp.get('message','')}")
    print(f"  {p2.agent_id}: {resp.get('decision','?')} - {resp.get('message','')}")

    if resp.get('decision') == 'accept':
        print(f"\n  >>> AGREEMENT REACHED")
        resolved = True
        break

if not resolved:
    print(f"\n  >>> ESCALATION NEEDED - no agreement after 2 rounds")

print("\n  Negotiation log:")
for l in log:
    print(f"    {l}")

STRATEGY 3: NEGOTIATION

--- Round 1 ---
  ThroughputAgent: {'action': 'HOLD_FOR_FAIRNESS_REVIEW', 'reasoning': "The FairnessAgent's concern regarding potential bias in the loan batch is valid. Proceeding to approval without a fairness review could lead to systemic discrimination or unfair treatment of certain demographic groups, which violates the core principle of equity. Therefore, the priority must be to ensure the decision is fair before any approval action is taken.", 'next_steps': ['Initiate a comprehensive fairness audit on Loan Batch 123.', 'Review historical lending data and demographic patterns for this batch.', 'Identify specific subgroups that may be disproportionately affected by the proposed approval.', 'Adjust the loan criteria or decision logic to mitigate identified biases.']}
  FairnessAgent: counter - The FairnessAgent's concern regarding potential bias in the loan batch is valid. However, proceeding to approval without a fairness review could lead to systemic discr

---
## Strategy 4: Game-Theoretic Resolution

In [7]:
print("=" * 55)
print("STRATEGY 4: GAME-THEORETIC")
print("=" * 55)

# Agent 4.1 - PayoffBuilder
pm = ask(f'''Two players: {p1.agent_id} (rows), {p2.agent_id} (columns).
Each can Push (insist) or Yield.
Payoffs as (row, column):
  Both Push: (-5, -5) deadlock
  Push/Yield: (+3, -2) row wins
  Yield/Push: (-2, +3) column wins
  Both Yield: (+1, +1) compromise
Output JSON: {{"matrix": {{"both_push":[-5,-5],"push_yield":[3,-2],"yield_push":[-2,3],"both_yield":[1,1]}}}}''')
print(f"  Payoff matrix: {pm.get('matrix', pm)}")

# Agent 4.2 - NashFinder
nf = ask(f'''Game matrix (Player1=rows, Player2=columns):
               Push          Yield
  Push        (-5,-5)       (+3,-2)
  Yield       (-2,+3)       (+1,+1)
Players: {p1.agent_id} is Player1, {p2.agent_id} is Player2.
Find Nash equilibrium.
Output JSON: {{"nash_eq": "...", "recommendation": "...", "reasoning": "..."}}''')
print(f"  Nash eq:     {nf.get('nash_eq', '?')}")
print(f"  Recommend:   {nf.get('recommendation', '?')}")
print(f"  Reasoning:   {nf.get('reasoning', '?')}")

STRATEGY 4: GAME-THEORETIC
  Payoff matrix: {'both_push': [-5, -5], 'push_yield': [3, -2], 'yield_push': [-2, 3], 'both_yield': [1, 1]}
  Nash eq:     Push
  Recommend:   ThroughputAgent should choose **Push**, while FairnessAgent should choose **Yield**. This combination constitutes the unique Nash Equilibrium of the game. Choosing Push guarantees a payoff of +3 (if FairnessAgent yields), whereas choosing Yield only guarantees +1 (if FairnessAgent yields) or -2 (if FairnessAgent pushes). Since FairnessAgent has an incentive to yield to avoid the lower payoff of -5, ThroughputAgent is incentivized to choose Push.
  Reasoning:   The game matrix shows that if Player 1 chooses Push, Player 2's best response is Yield (-2 > -5). If Player 1 chooses Yield, Player 2's best response is Push (+3 > +1). The intersection of these best responses is (Push, Yield). In this equilibrium, Player 1 gets +3 and Player 2 gets -2. Any deviation by Player 1 to Yield would result in a lower payoff (-2 vs +3)

---
## Summary

| Strategy | Mechanism | LLM agents | Outcome |
|---|---|---|---|
| Policy-Based | Deterministic priority rules | ConflictDetector, AuditLogger | Fairness approved |
| Hierarchical | Supervisor overrides based on risk | 2x RiskAssessor, Supervisor | Lower-risk plan approved |
| Negotiation | Peer proposals & responses | Proposer, Responder (x2) | Agreement or escalate |
| Game-Theoretic | Payoff matrix + Nash eq | PayoffBuilder, NashFinder | Equilibrium found |

Each agent uses a single-string prompt (no SystemMessage/HumanMessage list) with
`qwen3.5:2b-mlx (reasoning=False)`. This pattern is proven to work reliably for small models.